# Study API 入門：q-batch・Mixed・入力制約

このNotebookは実験候補をまとめて提案し、結果を登録する流れを示します。最初の候補生成にはGPの学習は不要です。

In [ ]:
import torch
from robotorchan.study import create_study
from robotorchan.optim.constraints.contracts import CandidateConstraints
from robotorchan.optim.domains.variable_space import MixedVariableSpace

## 1. q-batch候補の提案と結果登録

`ask_batch(3)` は3つのTrialをRUNNING状態で作成します。実験終了順に`tell`で結果を返せます。

In [ ]:
study = create_study(search_space={'x': (0.0, 1.0)}, seed=42)
trials = study.ask_batch(3)
for trial in reversed(trials):
    study.tell(trial, -(trial.params['x'] - 0.7) ** 2)
print(study.best_params, study.best_value)

## 2. 整数変数と制約付き初期候補

整数変数`count`を3以上に制限します。MixedVariableSpaceは既存のOptimizerと共通の契約です。

In [ ]:
variable_space = MixedVariableSpace(
    bounds=torch.tensor([[0.0, 1.0], [1.0, 4.0]], dtype=torch.double),
    integer_dims=(1,),
)
constraints = CandidateConstraints(
    inequality_constraints=((torch.tensor([1]), torch.tensor([1.0]), 3.0),),
)
mixed_study = create_study(
    search_space={'x': (0.0, 1.0), 'count': (1.0, 4.0)},
    variable_space=variable_space,
    constraints=constraints,
    seed=7,
)
mixed_trials = mixed_study.ask_batch(3)
for trial in mixed_trials:
    print(trial.number, dict(trial.params))
    assert trial.params['count'] in (3.0, 4.0)

## 3. q候補間制約

次の制約は2候補のxの和が1.5以上という意味です。候補を個別に評価するだけでは確認できないため、q-batch全体を検証します。

In [ ]:
joint_constraints = CandidateConstraints(
    inequality_constraints=((
        torch.tensor([[0, 0], [1, 0]]),
        torch.tensor([1.0, 1.0]),
        1.5,
    ),),
)
joint_study = create_study(
    search_space={'x': (0.0, 1.0)}, constraints=joint_constraints, seed=3,
)
joint_trials = joint_study.ask_batch(2)
print([dict(trial.params) for trial in joint_trials])
assert sum(trial.params['x'] for trial in joint_trials) >= 1.5 - 1e-7

## 注意点

初期候補は最大256回のSobol棄却サンプリングで生成します。可行領域が非常に狭い場合は候補生成が失敗することがあります。`Study.optimize()`は現時点では逐次実行です。モデルベースのMixed/q-batch最適化では、選択したモデル・獲得関数・Optimizerの互換性を確認してください。